In [2]:
import os
"""Analiza danych sprzedażowych w PySpark batch."""

from __future__ import annotations

from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, sum as spark_sum, desc


def main() -> None:
    """Uruchamia analizę wsadową w Spark."""
    # root_dir = Path(__file__).resolve().parents[1] # This line caused the NameError
    csv_path = Path("/content/data/sales.csv") # Directly specify the path

    spark = (
        SparkSession.builder.appName("BigDataBlock1SparkBatch")
        .master("local[*]")
        .getOrCreate()
    )

    df = spark.read.csv(str(csv_path), header=True, inferSchema=True)

    print("\n=== SPARK BATCH DEMO ===")
    print("\n1. Schemat danych:")
    df.printSchema()

    print("\n2. Podgląd danych:")
    df.show(5, truncate=False)

    print("\n3. Liczba rekordów:")
    print(df.count())

    print("\n4. Średnia wartość zamówienia:")
    df.select(avg("amount").alias("avg_amount")).show()

    print("\n5. Suma sprzedaży per kraj:")
    sales_by_country = df.groupBy("country").agg(spark_sum("amount").alias("total_sales"))
    sales_by_country.orderBy(desc("total_sales")).show(truncate=False)

    print("\n6. Kraj z największą sprzedażą:")
    sales_by_country.orderBy(desc("total_sales")).show(1, truncate=False)

    spark.stop()

main()


=== SPARK BATCH DEMO ===

1. Schemat danych:
root
 |-- order_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- order_date: timestamp (nullable = true)


2. Podgląd danych:
+---------+-------+-----------+------+-------------------+
|order_id |country|category   |amount|order_date         |
+---------+-------+-----------+------+-------------------+
|ORD-00001|Czechia|Books      |57.02 |2025-01-27 16:51:53|
|ORD-00002|Germany|Electronics|226.52|2025-01-10 22:49:51|
|ORD-00003|Czechia|Beauty     |148.67|2025-02-10 23:08:56|
|ORD-00004|Poland |Books      |158.67|2025-01-23 14:08:21|
|ORD-00005|Italy  |Beauty     |59.27 |2025-01-20 07:19:31|
+---------+-------+-----------+------+-------------------+
only showing top 5 rows

3. Liczba rekordów:
1000

4. Średnia wartość zamówienia:
+-----------------+
|       avg_amount|
+-----------------+
|755.5640700000005|
+-----------------+


5. Suma spr

In [3]:
"""Generator plików CSV do demonstracji Structured Streaming.

Skrypt co kilka sekund dopisuje nowy plik CSV do katalogu `stream_input`.
"""

from __future__ import annotations

import csv
import random
import time
from datetime import datetime
from pathlib import Path


COUNTRIES = ["Poland", "Germany", "France", "Spain", "Italy", "Czechia"]
CATEGORIES = ["Books", "Electronics", "Clothing", "Sports", "Beauty"]


def write_batch(batch_no: int, output_dir: Path, rows: int = 15) -> None:
    """Zapisuje pojedynczy plik z porcją zdarzeń sprzedażowych."""
    file_path = output_dir / f"batch_{batch_no:03d}.csv"

    with file_path.open("w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        for idx in range(rows):
            order_id = f"STR-{batch_no:03d}-{idx:03d}"
            country = random.choice(COUNTRIES)
            category = random.choice(CATEGORIES)
            amount = round(random.uniform(10.0, 900.0), 2)
            order_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            writer.writerow([order_id, country, category, amount, order_date])

    print(f"Written: {file_path.name}")


def main() -> None:
    """Generuje kolejne pliki wejściowe dla streamingu."""
    random.seed(123)

    # root_dir = Path(__file__).resolve().parents[1]
    csv_path = Path("/content/")
    output_dir = csv_path / "stream_input"
    output_dir.mkdir(parents=True, exist_ok=True)

    print("Generating streaming files...")
    for batch_no in range(1, 6):
        write_batch(batch_no=batch_no, output_dir=output_dir, rows=15)
        time.sleep(3)

    print("Done.")


main()

Generating streaming files...
Written: batch_001.csv
Written: batch_002.csv
Written: batch_003.csv
Written: batch_004.csv
Written: batch_005.csv
Done.


In [5]:
"""Demo Structured Streaming w PySpark.

Skrypt obserwuje katalog `stream_input` i agreguje sprzedaż per kraj.
"""

from __future__ import annotations

from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.functions import sum as spark_sum
from pyspark.sql.types import StructType, StringType, DoubleType


def main() -> None:
    """Uruchamia demo streamingu."""
    csv_path = Path("/content/")
    input_dir = csv_path / "stream_input"
    checkpoint_dir = csv_path / "stream_checkpoint"

    input_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    spark = (
        SparkSession.builder.appName("BigDataBlock1SparkStreaming")
        .master("local[*]")
        .getOrCreate()
    )

    schema = (
        StructType()
        .add("order_id", StringType())
        .add("country", StringType())
        .add("category", StringType())
        .add("amount", DoubleType())
        .add("order_date", StringType())
    )

    stream_df = (
        spark.readStream.schema(schema)
        .option("sep", ",")
        .csv(str(input_dir))
    )

    sales_by_country = stream_df.groupBy("country").agg(
        spark_sum("amount").alias("total_sales")
    )

    query = (
        sales_by_country.writeStream.outputMode("complete")
        .format("console")
        .option("truncate", False)
        .option("checkpointLocation", str(checkpoint_dir))
        .start()
    )

    print("Streaming started. Now run: python scripts/generate_streaming_files.py")
    query.awaitTermination()


if __name__ == "__main__":
    main()


Streaming started. Now run: python scripts/generate_streaming_files.py


ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [6]:
from __future__ import annotations

from pathlib import Path
import csv
import random
from datetime import datetime, timedelta

COUNTRIES = ["Poland", "Germany", "France", "Spain", "Italy", "Czechia"]
CATEGORIES = ["Books", "Electronics", "Clothing", "Sports", "Beauty", "Home"]
PAYMENTS = ["card", "blik", "transfer", "cash"]


def main() -> None:
    # root_dir = Path(__file__).resolve().parents[1]
    # data_dir = root_dir / "data"
    # data_dir.mkdir(parents=True, exist_ok=True)
    # output_path = data_dir / "transactions.csv"
    output_path = Path("/content/data/transactions.csv")


    random.seed(42)
    start = datetime(2025, 1, 1, 8, 0, 0)

    with output_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(
            [
                "transaction_id",
                "customer_id",
                "country",
                "category",
                "amount",
                "payment_method",
                "event_time",
            ]
        )

        for i in range(1, 50001):
            transaction_id = f"TX{i:06d}"
            customer_id = f"C{random.randint(1, 6000):05d}"
            country = random.choice(COUNTRIES)
            category = random.choice(CATEGORIES)
            amount = round(random.uniform(20, 800), 2)
            payment_method = random.choice(PAYMENTS)
            event_time = start + timedelta(minutes=i)

            writer.writerow(
                [
                    transaction_id,
                    customer_id,
                    country,
                    category,
                    amount,
                    payment_method,
                    event_time.isoformat(sep=" "),
                ]
            )

    print(f"Wygenerowano dane: {output_path}")


if __name__ == "__main__":
    main()


Wygenerowano dane: /content/data/transactions.csv


In [7]:
from __future__ import annotations

from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, count, desc, sum as spark_sum


def main() -> None:
    # root_dir = Path(__file__).resolve().parents[1]
    # csv_path = root_dir / "data" / "transactions.csv"
    csv_path = Path("/content/data/transactions.csv")

    spark = (
        SparkSession.builder.appName("BigDataBlock2Spark")
        .master("local[*]")
        .getOrCreate()
    )

    df = spark.read.csv(str(csv_path), header=True, inferSchema=True)

    print("\n=== SPARK ===")
    print("Schemat danych:")
    df.printSchema()

    print("\nPodgląd danych:")
    df.show(5, truncate=False)

    print("\n1. Liczba rekordów:")
    print(df.count())

    print("\n2. Średnia amount:")
    df.select(avg("amount").alias("avg_amount")).show()

    print("\n3. Suma sprzedaży per kraj:")
    df.groupBy("country").agg(spark_sum("amount").alias("total_sales")).orderBy(desc("total_sales")).show(truncate=False)

    print("\n4. Suma sprzedaży per kategoria:")
    df.groupBy("category").agg(spark_sum("amount").alias("total_sales")).orderBy(desc("total_sales")).show(truncate=False)

    print("\n5. Top 3 kraje:")
    df.groupBy("country").agg(spark_sum("amount").alias("total_sales")).orderBy(desc("total_sales")).show(3, truncate=False)

    print("\n6. Liczba transakcji per metoda płatności:")
    df.groupBy("payment_method").agg(count("transaction_id").alias("transactions")).orderBy(desc("transactions")).show(truncate=False)

    print("\n7. Transakcje > 400:")
    print(df.filter(df.amount > 400).count())

    print("\nKomentarz:")
    print("Spark wnosi rozproszony model wykonania, większy narzut startowy i styl pracy bliższy SQL oraz planowi wykonania.")

    spark.stop()


if __name__ == "__main__":
    main()



=== SPARK ===
Schemat danych:
root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- event_time: timestamp (nullable = true)


Podgląd danych:
+--------------+-----------+-------+--------+------+--------------+-------------------+
|transaction_id|customer_id|country|category|amount|payment_method|event_time         |
+--------------+-----------+-------+--------+------+--------------+-------------------+
|TX000001      |C05239     |Poland |Books   |598.41|blik          |2025-01-01 08:01:00|
|TX000002      |C01829     |Germany|Home    |99.95 |card          |2025-01-01 08:02:00|
|TX000003      |C04838     |Spain  |Books   |43.24 |blik          |2025-01-01 08:03:00|
|TX000004      |C01906     |Italy  |Beauty  |40.7  |blik          |2025-01-01 08:04:00|
|TX000005      |C05866     |Czech